# NUTDTS 816 Time Series Analysis
## L05 Stationarity, ACF/PACF, differencing, unit-root tests

Lab notebook for Chapter 3 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### 3.2 Three fundamental processes

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima_process import ArmaProcess
import tsdata
rng = np.random.default_rng(816); T = 300

def sim_ar1(phi, T, rng): 
    x = np.zeros(T)
    for t in range(1, T): x[t] = phi * x[t-1] + rng.normal()
    return x

series = {'White noise': rng.normal(size=T), 'AR(1), φ=0.5': sim_ar1(0.5, T, rng), 'AR(1), φ=0.95': sim_ar1(0.95, T, rng),
          'AR(1), φ=−0.7': sim_ar1(-0.7, T, rng), 'Random walk': np.cumsum(rng.normal(size=T)), 'Random walk with drift 0.2': np.cumsum(0.2 + rng.normal(size=T))}
fig, axes = plt.subplots(6, 2, figsize=(10, 12), gridspec_kw={'width_ratios': [2, 1]})
for i, (name, x) in enumerate(series.items()):
    axes[i, 0].plot(x, lw=0.9); axes[i, 0].set_title(name, loc='left')
    plot_acf(x, lags=30, ax=axes[i, 1], title=''); axes[i, 1].set_ylim(-1, 1)
_caption = 'Simulated processes (left) and their sample ACFs (right). Compare the fast decay of AR(1) with φ = 0.5, the slow decay with φ = 0.95, the alternating ACF for negative φ, and the trend-like ACF of the random walk.'

### 3.3 The partial autocorrelation function

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 5))
x = series['AR(1), φ=0.95']
plot_acf(x, lags=30, ax=axes[0, 0], title='AR(1), φ = 0.95: ACF (geometric decay)'); plot_pacf(x, lags=30, ax=axes[0, 1], title='PACF (cuts off after lag 1)')
x2 = ArmaProcess(ar=[1, -0.6, -0.3], ma=[1]).generate_sample(T, distrvs=rng.normal)
plot_acf(x2, lags=30, ax=axes[1, 0], title='AR(2), φ = (0.6, 0.3): ACF'); plot_pacf(x2, lags=30, ax=axes[1, 1], title='PACF (cuts off after lag 2)')
for ax in axes.flat: ax.set_ylim(-1, 1)
_caption = 'ACF and PACF of simulated AR(1) and AR(2) processes. The PACF cut-off identifies the order.'

### 3.4 Differencing

In [ ]:
ap = tsdata.airpassengers(); y = np.log(ap)
fig, axes = plt.subplots(4, 2, figsize=(10, 9), gridspec_kw={'width_ratios': [2, 1]})
steps = [('log x', y), ('∇ log x  (monthly growth)', y.diff()), ('∇₁₂ log x  (annual growth)', y.diff(12)), ('∇∇₁₂ log x', y.diff(12).diff())]
for i, (name, s) in enumerate(steps):
    s.dropna().plot(ax=axes[i, 0], lw=0.9, title=name); axes[i, 0].set_xlabel('')
    plot_acf(s.dropna(), lags=36, ax=axes[i, 1], title=''); axes[i, 1].set_ylim(-1, 1)
_caption = 'Log airline passengers through successive differencing. The monthly growth still has strong seasonality (ACF peaks at 12, 24, 36); the annual growth still has trend-like persistence; one of each leaves something close to stationary, with the ACF spikes at lags 1 and 12 that Chapter 5 will model.'

### 3.5 Unit-root tests

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
def unit_root_report(s, name, regression_adf='c', regression_kpss='c'):
    s = s.dropna()
    adf = adfuller(s, regression=regression_adf, autolag='AIC')
    kp = kpss(s, regression=regression_kpss, nlags='auto')
    print(f'{name:40s} ADF stat {adf[0]:7.2f}  p={adf[1]:.3f} (lags {adf[2]:2d})   |   KPSS stat {kp[0]:6.3f}  p={kp[1]:.3f}')

cpi = tsdata.nigeria_cpi(); fx = tsdata.nigeria_fx(); grid = tsdata.nigeria_grid(); oilp = tsdata.bonny_light()
print('ADF: H0 = unit root.   KPSS: H0 = stationary.   (KPSS p-values are capped at 0.01 / 0.10 by the table.)')
unit_root_report(np.log(oilp), 'log Bonny Light price (level)', 'ct', 'ct')
unit_root_report(np.log(oilp).diff(), 'log Bonny Light price (first difference)')
unit_root_report(np.log(fx), 'log NGN/USD (level)', 'ct', 'ct')
unit_root_report(np.log(fx).diff(), 'log NGN/USD (first difference)')
unit_root_report(np.log(cpi), 'log CPI (level)', 'ct', 'ct')
unit_root_report(np.log(cpi).diff(), 'log CPI (first difference = inflation)')
unit_root_report(np.log(cpi).diff().diff(), 'log CPI (second difference)')
unit_root_report(grid, 'grid generation (level)', 'ct', 'ct')
unit_root_report(grid.diff(), 'grid generation (first difference)')

## Exercises

1. Show that for the random walk $\mathrm{Corr}(X_t, X_{t-1}) = \sqrt{(t-1)/t}$ and compute it for $t = 10$ and $t = 1000$. What does this imply for the sample ACF of a long random walk?
2. For an AR(1) with $\phi = 0.9$ and $\sigma^2 = 1$, compute $\gamma(0)$, $\rho(1), \rho(6), \rho(12)$ and the half-life of a shock. Repeat for $\phi = -0.9$ and sketch the ACF.
3. Show that an MA(1) with $\theta = 2$ and one with $\theta = 0.5$ have the same ACF. Which is invertible? Write the first three $\pi$ weights of the invertible one's AR($\infty$) representation.
4. Simulate 200 observations of white noise, difference them, and plot the ACF of the result. Explain the spike at lag 1 and compute its theoretical value.

In [ ]:
# Your work here
